In [ ]:
!unzip src.zip

Archive:  src.zip
   creating: src/
   creating: src/dashboard/
   creating: src/data/
  inflating: src/data/download_vzcrash.py  
  inflating: src/data/eda.py         
  inflating: src/data/export_dataset_csv.py  
  inflating: src/data/preprocess_imu.py  
  inflating: src/data/synthetic_crashes.py  
  inflating: src/data/__init__.py    
   creating: src/data/__pycache__/
  inflating: src/data/__pycache__/eda.cpython-314.pyc  
  inflating: src/data/__pycache__/preprocess_imu.cpython-314.pyc  
  inflating: src/data/__pycache__/preprocess_severity.cpython-314.pyc  
  inflating: src/data/__pycache__/preprocess_supplementary.cpython-314.pyc  
  inflating: src/data/__pycache__/synthetic_crashes.cpython-314.pyc  
  inflating: src/data/__pycache__/__init__.cpython-314.pyc  
   creating: src/edge/
  inflating: src/edge/dispatch_alert.py  
  inflating: src/edge/inference.cpp  
   creating: src/features/
  inflating: src/features/feature_engineering.py  
  inflating: src/features/__init__.py  
 

In [ ]:
!unzip configs.zip

Archive:  configs.zip
   creating: configs/
  inflating: configs/config.yaml     


In [ ]:
!unzip models.zip

Archive:  models.zip
   creating: models/
   creating: models/checkpoints/
  inflating: models/checkpoints/best_bilstm.pth  
  inflating: models/checkpoints/best_lstm.pth  
  inflating: models/checkpoints/best_rash_classifier.pkl  
  inflating: models/checkpoints/rash_label_encoder.pkl  
  inflating: models/checkpoints/xgboost_rash_classifier.json  
   creating: models/onnx/
  inflating: models/onnx/intellicrash_lstm.onnx  
  inflating: models/onnx/intellicrash_lstm.onnx.data  


In [ ]:
!python download_vzcrash.py --token PUT_HF_TOKEN_HERE

--- IntelliCrash VZCrash Validation Downloader ---

🌐 Connecting to Hugging Face...
📢 Downloading a single Parquet shard directly (much faster and more stable)...
✅ Shard downloaded successfully to: /content/data/processed/vz_validation/data/train-00000-of-00011.parquet

📖 Loading Parquet file into memory with Pandas...
✅ Loaded 12,542 events from shard.

⏳ Processing events. Gathering 250 crash and 250 non-crash events...
Processing events:  97% 487/500 [00:03<00:00, 84.98it/s]
📊 Extracted 14,355 windows from 12542 events.
   Crash Windows: 1,225
   Non-Crash Windows: 13,130

⚡ Running Physics-Informed Feature Engineering (26 features per window)...

Extracting features:   0% 0/14355 [00:00<?, ?win/s]
Extracting features:   1% 158/14355 [00:00<00:09, 1573.97win/s]
Extracting features:   2% 343/14355 [00:00<00:08, 1733.88win/s]
Extracting features:   4% 524/14355 [00:00<00:07, 1765.55win/s]
Extracting features:   5% 711/14355 [00:00<00:07, 1803.82win/s]
Extracting features:   6% 892/14

In [ ]:
!python evaluate_vzcrash.py

--- IntelliCrash VZCrash Validation Evaluator ---
Loading VZCrash validation arrays...
📊 Loaded 14,355 validation windows.
   - Crash windows    : 1,225
   - Non-crash windows: 13,130

Preparing model inputs (depth-concatenating raw signals & expanded features)...

Loading trained Bi-LSTM model...
✅ Deployed model checkpoint loaded successfully!

Running neural network inference on validation windows...
⚠️ Warning: Optimal weights file not found. Using default 0.5:0.5

🔍 Optimizing Decision Thresholds (sweeping 0.01 to 0.99 to maximize F1-Score)...
   - Optimal LSTM Threshold   : 0.01
   - Optimal Physics Threshold: 0.10
   - Optimal Fusion Threshold : 0.03

  Real Crash Data Validation Results (VZCrash) — Default Threshold (0.50)
Configuration                       | Accuracy   | Recall   | FPR      | F1-Score   | AUC-ROC 
-----------------------------------------------------------------------------------------
1. Bi-LSTM Neural Path (1.0:0.0)    | 73.54    % | 13.14  % | 20.82  % | 7

In [ ]:
!python src/models/finetune_vzcrash.py

  IntelliCrash -- Domain Adaptation on VZCrash Real Crash Data
Device: cuda

Loading VZCrash validation arrays...
   Total windows: 14,355
   Crash: 1,225 | Non-crash: 13,130

Domain Adaptation Split:
   Fine-tune set: 1,435 windows (Crash: 122, Non-crash: 1,313)
   Test set:      12,920 windows (Crash: 1,103, Non-crash: 11,817)

Loading pre-trained Bi-LSTM model...
Pre-trained weights loaded successfully.

Running zero-shot inference (before fine-tuning)...

Fine-tuning on 10% of VZCrash real crash data...
   Learning rate: 1e-4 (low, to prevent catastrophic forgetting)
   Epochs: 5
   Batch size: 64
   Epoch 1/5 -- Loss: 1.9703, Accuracy: 68.4%
   Epoch 2/5 -- Loss: 1.4578, Accuracy: 69.2%
   Epoch 3/5 -- Loss: 1.2910, Accuracy: 68.6%
   Epoch 4/5 -- Loss: 1.2038, Accuracy: 69.8%
   Epoch 5/5 -- Loss: 1.1470, Accuracy: 67.8%
Fine-tuning complete!

Running fine-tuned inference on 90% held-out test set...

  ZERO-SHOT Results (Before Fine-Tuning) -- Optimized Thresholds
Configuration (

In [ ]:
import os
import zipfile
from google.colab import files

# 1. Define files to zip and download
files_to_download = [
    # Reports
    "outputs/reports/vzcrash_zero_shot_report.csv",
    "outputs/reports/vzcrash_fine_tuned_report.csv",

    # Fine-Tuned Neural Checkpoint
    "models/checkpoints/best_bilstm_vzcrash_finetuned.pth",

    # Extracted validation CSV
    "data/processed/vz_validation/vz_features.csv",

    # Processed NumPy arrays (for future code/eval validations)
    "data/processed/vz_validation/vz_X.npy",
    "data/processed/vz_validation/vz_y.npy",
    "data/processed/vz_validation/vz_features.npy",
    "data/processed/vz_validation/vz_csi.npy"
]

zip_name = "vzcrash_local_outputs.zip"

# 2. Package into a zip archive
print("Packaging all VZCrash output files...")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in files_to_download:
        if os.path.exists(file_path):
            zipf.write(file_path)
            print(f"  Added: {file_path}")
        else:
            print(f"  Warning: {file_path} not found")

# 3. Trigger browser download
print("\nTriggering download window...")
files.download(zip_name)

Packaging all VZCrash output files...
  Added: outputs/reports/vzcrash_zero_shot_report.csv
  Added: outputs/reports/vzcrash_fine_tuned_report.csv
  Added: models/checkpoints/best_bilstm_vzcrash_finetuned.pth
  Added: data/processed/vz_validation/vz_features.csv
  Added: data/processed/vz_validation/vz_X.npy
  Added: data/processed/vz_validation/vz_y.npy
  Added: data/processed/vz_validation/vz_features.npy
  Added: data/processed/vz_validation/vz_csi.npy

Triggering download window...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
import pandas as pd

# Load the raw 3D array of windows (N, 200, 6)
# Columns: [accele_x, accele_y, gyro_z, accele_x_filtered, accele_y_filtered, gyro_z_filtered]
X = np.load("data/processed/vz_validation/vz_X.npy")
y = np.load("data/processed/vz_validation/vz_y.npy")

# Filter for the first 5,000 samples (~25 crash windows) to keep file small and fast
num_windows_to_export = 250
stacked_rows = []

for win_idx in range(min(num_windows_to_export, len(X))):
    window_data = X[win_idx]
    label = y[win_idx]

    for t_idx in range(len(window_data)):
        stacked_rows.append({
            "window_index": win_idx,
            "sample_index": t_idx,
            "time_seconds": t_idx * 0.01,
            "accel_x_g": window_data[t_idx, 0],
            "accel_y_g": window_data[t_idx, 1],
            "gyro_z_rad_s": window_data[t_idx, 2],
            "accel_x_filtered_g": window_data[t_idx, 3],
            "accel_y_filtered_g": window_data[t_idx, 4],
            "gyro_z_filtered_rad_s": window_data[t_idx, 5],
            "is_crash_window": label
        })

df_raw = pd.DataFrame(stacked_rows)
output_path = "outputs/reports/vzcrash_raw_telemetry_excel.csv"
df_raw.to_csv(output_path, index=False)
print(f"Raw telemetry CSV saved successfully (size: {len(df_raw)} rows) at: {output_path}")

# Trigger direct download
from google.colab import files
files.download(output_path)


Raw telemetry CSV saved successfully (size: 50000 rows) at: outputs/reports/vzcrash_raw_telemetry_excel.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>